# mvTCR Preprocessing
mvTCR uses a specific format to handle single-cell data, which is based on AnnData objects. If not otherwise stated, we follow the speficition from Scanpy [1] and Scirpy [2]. However, we need some additional information to utilize all functions of mvTCR. In this notebook, we introduce the mvTCR preprocessing pipeline, which adds the required information to the corresponding place in the AnnData object.

All experiments in our paper where conducted on Datasets:
- after Quality Control (cell filtering, doublet detection, ...)
- with normalized and log+1 transformed count data

The pipeline assumes that these steps have already been performed. For further reference, please see Luecken et al [3].

If you know what you are doing: different normalization, log-stabilizing transformations, etc. can also be used, but need to be handled with care!


[1] Wolf, F. A., Angerer, P. & Theis, F. J. Scanpy: large-scale single-cell gene expression data analysis. Genome biology 19, 1–5 (2018).

[2] Sturm, G. et al. Scirpy: a scanpy extension for analyzing single-cell t-cell receptor-sequencing data. Bioinformatics 36, 4817–4818 (2020).

[3] Luecken, M. D. & Theis, F. J. Current best practices in single-cell rna-seq analysis: a tutorial.
Molecular systems biology 15, e8746 (2019).

## Prerequisits

The preprocessing pipeline is showcased on the dataset from Stephenson et al. [4], which can be readily downloaded from:

- https://covid19.cog.sanger.ac.uk/submissions/release1/haniffa21.processed.h5ad
- https://www.ebi.ac.uk/biostudies/files/E-MTAB-10026/TCR_merged-Updated.tsv

and is already quality-controled. 

[4] Stephenson, E. et al. Single-cell multi-omics analysis of the immune response in covid-19. Nature medicine 27, 904–916 (2021).


The mvTCR preprocessing pipeline is taylored for mvTCR-usage and handles the encoding of clonotypes and conditional variables in the required format. However, it is necessary that the adata object is already log-normalized, subsetted to highly variable genes and contains scirpy-encoded TCR information. We demonstrate these steps below.

In [1]:
import sys
print(sys.executable)
print(sys.prefix)

/ihome/ylee/yiz133/.conda/envs/mvTCR2/bin/python
/ihome/ylee/yiz133/.conda/envs/mvTCR2


In [2]:
import scanpy as sc
import scirpy as ir
import pandas as pd
import awkward as ak

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

/ihome/ylee/yiz133/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: cannot import name 'METRIC_MAPPING64' from 'sklearn.metrics._dist_metrics' (/ihome/ylee/yiz133/.conda/envs/mvTCR2/lib/python3.11/site-packages/sklearn/metrics/_dist_metrics.cpython-311-x86_64-linux-gnu.so)

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# import muon as mu
# mdata = mu.read("/ix1/ylee/Yifan_Zhang/Code_data/data_EAE/anndata/all_Common_DEtop2500.h5mu")
# processed data not working

# mdata = mu.read("/ix1/ylee/Yifan_Zhang/Code_data/data_EAE/anndata/all_Common.h5mu")    # work
# adata = mdata['gex']
# adata_tcr = mdata['airr']


In [ ]:
adata = sc.read_h5ad('v7_avidity.h5ad')
adata

We will load the transcriptome data. To speed up runtime, we will downsample the data to two patients.

In [ ]:
selected_sample = ['5_3']
adata = adata[adata.obs['mouse_id'].isin(selected_sample)].copy()

Before starting, we take the raw expression counts matrix, total-count normalize it to 10,000 reads per cell to correct for differences in library-size, and logarithmize it:

Next we add the required TCR information as scirpy formatted covariates in the obs matrix:

Lets index the chains

mvTCR requires paired data between TCR and GEX. Therefore, we remove all samples without a TRA or TRB CDR3 region.

## mvTCR preprocessing

In [ ]:
from mvtcr.utils_preprocessing import Preprocessing

## All-in-one Pipeline

After we have a fitting dataset containing scirpy-encoded TCR information and expression data we can use mvTCR's preprocessing methods to further bring our data into shape.
The preprocessing pipeline is the fast way to do that your data. 

This features (in order):

- Checks for:
    - Normalization & log transformation checks (experimental)
    - "Reasonable" number of highly variable genes check (500 < n < 5000)
    - Scirpy VDJ gene usage information check
- Encoding of clonotypes
- Encoding of TCR
- One-Hot encoding of conditional variables

The required parameters and expected outputs of each step are explained in detail in the piece by piece preprocessing section below.

Please note: If you are using MuData object please convert it into an AnnData object where the AIRR data is stored in obsm under "airr". We offer a conversion function to do so at the end of this notebook.

In [ ]:
Preprocessing.preprocessing_pipeline(adata, 
                                     clonotype_key_added='clonotype', 
                                     airr_name='junction_aa', 
                                     cond_vars=['mouse_id'])

# Piece by Piece Preprocessing

All the features inside the pipeline can be executed seperately as well, to perform a step-by-setp preprocessing or only specific methods.

Make sure to freshly load the data if you have used the pipeline.

### Checking if adata is in a mvTCR compatible shape

In [ ]:
Preprocessing.check_if_valid_adata(adata)

### Encoding clonotypes with Scirpy

For training the shared embedding, we advise oversampling rare clonotypes. This avoids the model overfitting to few selected TCR sequences from highly expanded clonotypes. Therefore, we need to add a clonotype label to adata.obs. Here, we define a unique clonotype via Scirpy as having exactly the same CDR3 sequence in TRA and TRB chains.

In [ ]:
Preprocessing.encode_clonotypes(adata, key_added='clonotype')

adata.obs.clonotype.value_counts()

### Adding TCR encoding

Next, we encode the TCR sequence numerically to adata.obsm. Here, we need to provide the name of the column storing the CDR3a and CDR3b. Additionally, we need to specificy the padding paremter (which if set to None uses the maximal CDR3 sequence length as default). If you plan to add new data in the future via a pretrained model, you might want to add some safety margin.

In [ ]:
Preprocessing.encode_tcr(adata, 
                         airr_name='junction_aa', 
                         alpha_label_key='alpha_seq', 
                         alpha_length_key='alpha_len',
                         beta_label_key='beta_seq', 
                         beta_length_key='beta_len')

adata.obsm['beta_seq'][:5]

### Adding conditional variables

Conditioning your model partially removes the effect from a specified condition. We can add conditional variables for e.g. donor, to avoid batch effects over multiple samples. The encoded variable carries the suffix "_ohe" so it can be distinguished from its original part. 

In [ ]:
Preprocessing.encode_conditional_var(adata, column_id='mouse_id')
adata.obsm['mouse_id_ohe']

### Creating training and validation splits

The splitting improves the data spliting into training and validation sets by two properties:
- Stratified splitting: balance a label of interest (normally a variable to be predicted, e.g. antigen specificity) so the label distribution is roughly the same in both sets.
- Avoid training data leakage into validation: used for clonotypes, to ensure that each clonotype is observed only during training or validation.

In [ ]:
train, val = Preprocessing.stratified_group_shuffle_split(adata, stratify_col='full_clustering', group_col='clonotype', test_size=0.2, random_seed=42)

adata.obs['set'] = 'train'
adata.obs.loc[val, 'set'] = 'val'

adata.obs["set"].value_counts()

Alternatively group splitting is available by itself with:

In [ ]:
train, val = Preprocessing.group_shuffle_split(adata, group_col='clonotype', test_size=0.2, random_seed=42)

adata.obs['set'] = 'train'
adata.obs.loc[val, 'set'] = 'val'

adata.obs["set"].value_counts()

### Finish. You are all set and done to use mvTCR! Save your data!

In [ ]:
path_out = '../data/preprocessed/haniffa_test_new.h5ad'
adata.write_h5ad(path_out, compression='gzip')

### Conversion: AnnData <> MuData

In [ ]:
#First lets generate a MuData object 
mdata = Preprocessing.adata_to_mudata(adata, obs_cols=[], obsm_cols=["airr", "chain_indices"], uns_cols=["chain_indices"], 
                                      keep_obs_cols=False, keep_obsm_cols=False, keep_uns_cols=False)
mdata

In [ ]:
#And lets change back to adata
adata = Preprocessing.mudata_to_adata(mdata, mudata_gex_key='gex', mudata_airr_key='airr')
adata

In order for mvTCR to work you need to have a adata obj with the gene expression in X, features in obs and tcr data in obsm (key: 'airr'). Refer to the 00_tutorial to see how we built a working dataset from scratch.